# XLCoST boolean pipeline — Colab

Corpus build → occurrence extraction → activation extraction → probe + baselines,
all via the committed CLIs (PROTOCOL.md v1.0). Runtime → **GPU** (L4 on Pro).

Everything is resumable: re-running the extraction cell continues where it stopped.

In [ ]:
BRANCH = "nolan-boolean"
LANGUAGE = "Python"          # Python | Java | Javascript | PHP
SPLIT = "train"
MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B"

import os, pathlib
if not pathlib.Path("mech-interp-coding-llms").exists():
    !git clone -q -b {BRANCH} https://github.com/nolanlwin/mech-interp-coding-llms.git
%cd mech-interp-coding-llms
!pip install -q transformers==5.8.0 tree_sitter "tree-sitter-java>=0.23.5" "tree-sitter-go>=0.25.0" \
  "tree-sitter-javascript>=0.25.0" "tree-sitter-php>=0.24.1" "tree-sitter-ruby>=0.23.1" \
  scikit-learn scipy huggingface_hub
# Optional: HF token from Colab secrets for faster downloads
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    pass

In [ ]:
# GATE — do not extract if this fails (PROTOCOL.md §3.2)
!python scripts/tokenizer_gate.py run --models {MODEL_ID}

In [ ]:
lang_slug = LANGUAGE.lower().replace("++", "pp").replace("#", "sharp")
!python scripts/xlcost_data.py build --language {LANGUAGE} --split {SPLIT} --out-dir data/xlcost
!python scripts/xlcost_occurrences.py extract \
  --input data/xlcost/{lang_slug}_{SPLIT}.jsonl \
  --output outputs/xlcost_occ/{lang_slug}_{SPLIT}.jsonl

In [ ]:
# Activation extraction — one forward per program, fp16 memmap store.
# ~16.7k Python occurrences on an L4: well under an hour. Resumable.
model_slug = MODEL_ID.split("/")[-1].lower().replace(".", "").replace("-", "")
STORE = f"outputs/activations_xlcost/{lang_slug}_{SPLIT}_{model_slug}"
!python scripts/extract_activations.py run \
  --canonical data/xlcost/{lang_slug}_{SPLIT}.jsonl \
  --occurrences outputs/xlcost_occ/{lang_slug}_{SPLIT}.jsonl \
  --model-id {MODEL_ID} --out-dir {STORE} --log-every 1000

In [ ]:
# Probe (problem-grouped split, control task, CIs) + model-free baselines
!python scripts/probe.py run --store {STORE} \
  --split-policy repo --allow-class-drop --control-task --occurrence-cap 2000 \
  --output outputs/probe_results/{lang_slug}_{SPLIT}_{model_slug}_problem.json
!python scripts/baselines.py run \
  --occurrences outputs/xlcost_occ/{lang_slug}_{SPLIT}.jsonl \
  --canonical data/xlcost/{lang_slug}_{SPLIT}.jsonl \
  --split-policy repo \
  --output outputs/probe_results/{lang_slug}_{SPLIT}_baselines.json

In [ ]:
# Persist to Drive: results always; the store only for models we keep
# (PROTOCOL: persist small models, extract-probe-discard the big ones).
from google.colab import drive
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/mech-interp/xlcost"
!mkdir -p {DEST}/probe_results && cp -r outputs/probe_results/* {DEST}/probe_results/
!mkdir -p {DEST}/stores && cp -r {STORE} {DEST}/stores/